# Splitly growth analysis

Question from the board: is the viral coefficient (K) really close to 1?

Data: 5 raw CSV files, Jan 2025 to Jun 2026, extracted on 30 June 2026. Amounts in euros.

Sections: 1 Cleaning, 2 Helper columns, 3 Funnel and loop, 4 Viral coefficient, 5 Segments, 6 Unit economics, 7 What changed, 8 North Star Metric, 9 Growth model, 10 A/B test, 11 Answer to the board.

## 0. Load the raw files

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

In [ ]:
DATA = "data/"   # folder with the csv files, change it if needed

users_raw   = pd.read_csv(DATA + "users.csv")
invites_raw = pd.read_csv(DATA + "invites.csv")
rewards_raw = pd.read_csv(DATA + "referral_rewards.csv")
spend       = pd.read_csv(DATA + "marketing_spend.csv")
ratings     = pd.read_csv(DATA + "app_store_ratings.csv")

In [ ]:
# key dates of the case
START = pd.Timestamp("2025-01-01")           # first day of the data
END = pd.Timestamp("2026-06-30")             # extraction date
REFERRAL_START = pd.Timestamp("2025-11-01")  # double sided referral bonus starts
SYNC_START = pd.Timestamp("2026-03-01")      # contact sync and auto SMS start

In [ ]:
sizes = {"users": users_raw, "invites": invites_raw, "rewards": rewards_raw, "spend": spend, "ratings": ratings}
for name, df in sizes.items():
    print(f"{name:10s} {df.shape[0]:>8,} rows  {df.shape[1]} columns")
users_raw.head()

## 1. Data cleaning

First we explore the data (missing values, labels, dates), then we clean it step by step. Every rule is stored in a small list so we can print a report at the end.

### 1.1 First look: what is missing?

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(16, 3))
for ax, (name, df) in zip(axes, sizes.items()):
    df.isna().sum().plot(kind="barh", ax=ax, title=name)
plt.tight_layout()
plt.show()

In [ ]:
# do the missing values go together? share of missing values by source
miss_by_source = users_raw.drop(columns="source").isna().groupby(users_raw.source).mean()
display(miss_by_source)

plt.imshow(miss_by_source.values, cmap="Reds", vmin=0, vmax=1, aspect="auto")
plt.xticks(range(miss_by_source.shape[1]), miss_by_source.columns, rotation=45, ha="right")
plt.yticks(range(miss_by_source.shape[0]), miss_by_source.index)
plt.colorbar(label="share missing")
plt.title("Missing values in users, by source")
plt.show()

In [ ]:
# in invites, invitee_user_id should be empty exactly when the invite was not accepted
display(invites_raw.isna().groupby(invites_raw.accepted).mean())

Findings: `inviter_user_id` is empty only for organic and paid users (expected). `country` is missing in every source, so it is a real data gap. `invitee_user_id` is empty only when the invite was not accepted (expected).

In [ ]:
cleaning_log = []

def note(step, table, problem, action, rows):
    cleaning_log.append({"step": step, "table": table, "problem": problem, "action": action, "rows": int(rows)})

### 1.2 users: dates

In [ ]:
users = users_raw.copy()
text = users["signup_date"].astype(str)
is_iso = text.str.match(r"^\d{4}-\d{2}-\d{2}$")
print("rows with a dd/mm/yyyy date:", (~is_iso).sum())

In [ ]:
# is it day first or month first? look at the two numbers of the slash dates
print("max first number :", text[~is_iso].str[0:2].astype(int).max(), "(above 12, so it is the day)")
print("max middle number:", text[~is_iso].str[3:5].astype(int).max())

In [ ]:
date_iso = pd.to_datetime(text.where(is_iso), format="%Y-%m-%d")
date_slash = pd.to_datetime(text.where(~is_iso), format="%d/%m/%Y")
users["signup_date"] = date_iso.fillna(date_slash)
users["last_active_date"] = pd.to_datetime(users["last_active_date"])
note("U1", "users", "signup_date mixes yyyy-mm-dd and dd/mm/yyyy", "parse each format separately", (~is_iso).sum())

In [ ]:
# impossible dates (placeholders like 1900, 1970, 2099)
bad_date = (users.signup_date < START) | (users.signup_date > END)
display(users.loc[bad_date, ["user_id", "signup_date", "source", "last_active_date"]])

bad_user_ids = set(users.loc[bad_date, "user_id"])
note("U2", "users", "signup_date outside the data period", "drop the user", bad_date.sum())
users = users[~bad_date]

### 1.3 users: test accounts and missing country

In [ ]:
is_test = (users.source == "internal") | users.user_id.str.startswith("QA_")
display(users[is_test].head(3))

In [ ]:
test_ids = set(users.loc[is_test, "user_id"])
note("U3", "users", "internal QA accounts (250 splits each)", "drop them and their invites", is_test.sum())
users = users[~is_test]

n_missing = users.country.isna().sum()
users["country"] = users["country"].fillna("Unknown")
note("U4", "users", "country is missing", "fill with Unknown, keep the row", n_missing)

In [ ]:
# consistency checks, 0 rows means the check passed
checks = {
    "duplicated user_id": users.user_id.duplicated().sum(),
    "last_active before signup": (users.last_active_date < users.signup_date).sum(),
    "splits_first_30d above 100": (users.splits_first_30d > 100).sum(),
    "activated does not match splits": (((users.activated == 0) & (users.splits_first_30d > 0)) |
                                        ((users.activated == 1) & (users.splits_first_30d == 0))).sum(),
    "source does not match inviter": (((users.source == "invite") & users.inviter_user_id.isna()) |
                                      ((users.source != "invite") & users.inviter_user_id.notna())).sum(),
}
for name, n in checks.items():
    note("U5", "users", name, "check only", n)

n_inviter_removed = users.inviter_user_id.isin(bad_user_ids | test_ids).sum()
note("U6", "users", "invited users whose inviter was removed", "keep the user, inviter unknown", n_inviter_removed)
print("users after cleaning:", len(users))

### 1.4 invites

In [ ]:
invites = invites_raw.copy()
invites["sent_date"] = pd.to_datetime(invites["sent_date"], errors="coerce")
print("unparsed dates:", invites.sent_date.isna().sum())
print("range:", invites.sent_date.min().date(), "to", invites.sent_date.max().date())

n_dup = invites.duplicated().sum()
invites = invites.drop_duplicates()
note("I1", "invites", "exact duplicate rows", "drop_duplicates", n_dup)

In [ ]:
invites.channel.value_counts().plot(kind="bar", title="Invite channel labels (before cleaning)")
plt.show()

In [ ]:
channel_map = {"WhatsApp": "whatsapp", "wa": "whatsapp", "SMS": "sms"}
n_labels = invites.channel.isin(channel_map.keys()).sum()
invites["channel"] = invites["channel"].replace(channel_map)
note("I2", "invites", "same channel with different labels", "map to 4 channels", n_labels)
print(invites.channel.value_counts())

In [ ]:
is_test_invite = invites.inviter_user_id.isin(test_ids)
note("I3", "invites", "invites sent by QA accounts", "drop", is_test_invite.sum())
invites = invites[~is_test_invite]

bad_invitee = invites.invitee_user_id.isin(bad_user_ids)
note("I4", "invites", "accepted invite whose invitee was removed", "drop", bad_invitee.sum())
invites = invites[~bad_invitee]

n_acc = (((invites.accepted == 1) & invites.invitee_user_id.isna()) |
         ((invites.accepted == 0) & invites.invitee_user_id.notna())).sum()
note("I5", "invites", "accepted flag does not match invitee", "check only", n_acc)
note("I6", "invites", "invite_id duplicated after I1", "check only", invites.invite_id.duplicated().sum())

June 2026 looks incomplete. Let's see the invites per day.

In [ ]:
daily = invites.groupby("sent_date").size()
daily["2026-04-01":].plot(figsize=(10, 3), title="Invites per day, since April 2026")
plt.show()
print("average per day in May 2026:", round(daily["2026-05-01":"2026-05-31"].mean()))
print(daily.tail(5))

In [ ]:
LAST_FULL = pd.Timestamp("2026-05-31")   # last complete month
n_june = (invites.sent_date > LAST_FULL).sum()
note("I7", "invites and users", "June 2026 is incomplete (looks like extraction lag)",
     "keep rows but exclude June from rates, trends and the model", n_june)

### 1.5 rewards, spend and ratings

In [ ]:
rewards = rewards_raw.copy()
rewards["paid_date"] = pd.to_datetime(rewards["paid_date"], errors="coerce")

removed = bad_user_ids | test_ids
bad_reward = rewards.invitee_user_id.isin(removed) | rewards.inviter_user_id.isin(removed)
note("R1", "rewards", "reward linked to a removed user", "drop", bad_reward.sum())
rewards = rewards[~bad_reward]

In [ ]:
r = rewards.merge(users[["user_id", "signup_date", "activated"]], left_on="invitee_user_id", right_on="user_id", how="left")
note("R2", "rewards", "reward paid before signup", "check only", (r.paid_date < r.signup_date).sum())
note("R3", "rewards", "duplicate invitee or cost different from 10 euros", "check only",
     rewards.invitee_user_id.duplicated().sum() + (rewards.total_cost_eur != 10).sum())

# the rule says the reward is paid at the first split, so activated should always be 1
rewards["invitee_activated"] = r.activated.values
note("R4", "rewards", "reward paid but the invitee never made a split",
     "keep (money was paid), flag it and measure the leak", (r.activated == 0).sum())

In [ ]:
spend["month"] = pd.to_datetime(spend["month"])
print("channels in spend:", spend.channel.unique(), "| months:", spend.month.nunique(), "| negatives:", (spend.spend_eur < 0).sum())
note("M1", "marketing_spend", "only one channel (paid ads), 18 months", "check only, spend is paid acquisition only", 0)

ratings["month"] = pd.to_datetime(ratings["month"])
all_months = pd.date_range("2025-01-01", "2026-06-01", freq="MS")
missing_months = [m.strftime("%Y-%m") for m in all_months if m not in set(ratings.month)]
ratings = ratings.set_index("month").reindex(all_months).rename_axis("month").reset_index()
note("A1", "app_store_ratings", f"missing month {missing_months}", "re-insert as NaN, not invented", len(missing_months))

### 1.6 Cleaning report

In [ ]:
log = pd.DataFrame(cleaning_log)
print("CLEANING REPORT")
for _, row in log.iterrows():
    print(f"{row.step:3s} {row.table:18s} {row.rows:>7,}  {row.problem} -> {row.action}")

before = [len(users_raw), len(invites_raw), len(rewards_raw)]
after = [len(users), len(invites), len(rewards)]
print()
print(pd.DataFrame({"before": before, "after": after}, index=["users", "invites", "rewards"]))

What matters most:
* Biggest impact on the numbers: 7,616 invites with inconsistent channel labels, 1,993 duplicated invites, 1,439 users with day first dates, and June 2026 incomplete (14,604 invites excluded from rates).
* Removed: 12 QA accounts (250 splits each, 600 invites, none accepted) and 6 users with placeholder dates.
* Kept on purpose: 1,856 users without country (labelled Unknown) and 6,134 rewards paid to invitees who never made a split. The money was really paid, so we measure it in section 3.3.
* Checks that passed: duplicated users, last_active before signup, activated vs splits, source vs inviter, reward cost, reward before signup.
* Limits we cannot fix: no revenue data (LTV needs an assumption), no city column, only one marketing channel in the spend file, July 2025 missing in ratings.

## 2. Helper columns

* activated: made at least one split.
* retained_30: still active 30 or more days after signup. Only users who signed up 30 days before the extraction can be judged.
* good user: activated and retained_30. This is our quality filter.
* period: 1 before the referral bonus (Jan to Oct 2025), 2 referral bonus (Nov 2025 to Feb 2026), 3 bonus plus contact sync (Mar 2026 on).

In [ ]:
users["signup_month"] = users.signup_date.dt.to_period("M")
users["life_days"] = (users.last_active_date - users.signup_date).dt.days
users["retained_30"] = (users.life_days >= 30).astype(int)
users["good_user"] = ((users.activated == 1) & (users.retained_30 == 1)).astype(int)
users["obs_30"] = users.signup_date <= (END - pd.Timedelta(days=30))

In [ ]:
def get_period(dates):
    return np.select([dates < REFERRAL_START, dates < SYNC_START],
                     ["1_before_referral", "2_referral_bonus"], default="3_referral_plus_sync")

users["period"] = get_period(users.signup_date)
invites["sent_month"] = invites.sent_date.dt.to_period("M")
invites["period"] = get_period(invites.sent_date)

In [ ]:
# analysis data: complete months only (Jan 2025 to May 2026)
full = users[users.signup_date <= LAST_FULL].copy()
inv_full = invites[invites.sent_date <= LAST_FULL].copy()
print("users used:", len(full), "| invites used:", len(inv_full), "| users with 30 days of history:", full.obs_30.sum())

## 3. Funnel and growth loop: where does value leak?

Splitly has two engines. A funnel fed by money (ads) and a loop fed by users: an active user invites friends, the friend signs up, makes a first split, and becomes an active user who invites more.

In [ ]:
pd.options.display.float_format = "{:,.3f}".format   # show numbers with 3 decimals

monthly_signups = full.pivot_table(index="signup_month", columns="source", values="user_id", aggfunc="count")
monthly_signups["total"] = monthly_signups.sum(axis=1)
monthly_signups["invite_share"] = monthly_signups["invite"] / monthly_signups["total"]
display(monthly_signups)

In [ ]:
monthly_signups[["organic", "paid", "invite"]].plot(kind="bar", stacked=True, figsize=(11, 4), title="New users per month by source")
plt.show()

In [ ]:
act_month = full.pivot_table(index="signup_month", columns="source", values="activated", aggfunc="mean")
act_month.plot(figsize=(10, 4), title="Activation rate by signup month and source")
plt.axvline(pd.Period("2025-11"), color="gray", ls="--")
plt.axvline(pd.Period("2026-03"), color="red", ls="--")
plt.show()

* Paid and organic grow smoothly (about 3% per month) and did not change when the product changed, so they work as a control group.
* The explosion is the invite channel: about 650 signups per month before the bonus, 4,000 after the bonus, 5,500 after the contact sync. Invites are now 55 to 62% of new users.
* Quality fell: invitee activation dropped from 62% to 29 to 32%, while paid and organic stay near 55%.

### 3.1 Loop funnel: invites, signups, activated, retained

In [ ]:
acc = inv_full[inv_full.accepted == 1].merge(
    users[["user_id", "activated", "retained_30", "good_user", "obs_30"]],
    left_on="invitee_user_id", right_on="user_id", how="left")

def funnel(df_inv, df_acc, by):
    out = pd.DataFrame({"invites_sent": df_inv.groupby(by).size()})
    out["signups"] = df_acc.groupby(by).size()
    out["activated"] = df_acc.groupby(by).activated.sum()
    out["accept_rate"] = out.signups / out.invites_sent
    out["activation_rate"] = out.activated / out.signups
    # retention only for invitees with 30 days of history
    a30 = df_acc[(df_acc.obs_30) & (df_acc.activated == 1)]
    out["retained_given_activated"] = a30.groupby(by).retained_30.mean()
    out["good_per_1000_invites"] = 1000 * out.accept_rate * out.activation_rate * out.retained_given_activated
    return out

f_period = funnel(inv_full, acc, "period")
display(f_period)

In [ ]:
steps = pd.DataFrame({
    "invites": 1000,
    "signups": 1000 * f_period.accept_rate,
    "activated": 1000 * f_period.accept_rate * f_period.activation_rate,
    "good users": f_period.good_per_1000_invites})
steps.T.plot(kind="bar", figsize=(9, 4), title="Out of 1000 invites sent, how many reach each step?")
plt.xticks(rotation=0)
plt.show()

Out of 1,000 invites: 74 good users before the bonus, 28 with the bonus, 11 with bonus plus sync. An invite became 7 times less efficient. Three leaks:
1. Invite to signup: acceptance went 25% to 39% (bonus) to 14% (sync floods people with automatic SMS).
2. Signup to first split: activation halved (62% to 29 to 32%). Many people sign up for the reward.
3. Activated to active at day 30: also halved (49% to 24%).

### 3.2 Funnel by invite channel (period 3, same period for a fair comparison)

In [ ]:
p3_inv = inv_full[inv_full.period == "3_referral_plus_sync"]
p3_acc = acc[acc.invite_id.isin(p3_inv.invite_id)]
f_channel = funnel(p3_inv, p3_acc, "channel")
display(f_channel)

print("share of invites that are auto SMS:", round((p3_inv.channel == "auto_sms_contact_sync").mean(), 3))
print("share of accepted invites from auto SMS:", round((p3_acc.channel == "auto_sms_contact_sync").mean(), 3))

* Auto SMS is 67% of invites but only 12% of accepted ones (acceptance 2.5% vs 40% WhatsApp and 43% link).
* Good users per 1,000 invites: WhatsApp 38, SMS 20, link 19, auto SMS 2.
* Link has high acceptance but the lowest activation (18%), a typical sign of reward hunting or links shared in open groups.

### 3.3 Reward leakage

In [ ]:
rw = rewards.copy()
rw["month"] = rw.paid_date.dt.to_period("M")
leak = rw.groupby("month").agg(rewards=("reward_id", "count"), cost_eur=("total_cost_eur", "sum"),
                              paid_without_split=("invitee_activated", lambda s: (s == 0).sum()))
leak["leak_eur"] = leak.paid_without_split * 10
leak["leak_share"] = leak.paid_without_split / leak.rewards
display(leak)

In [ ]:
print("total reward cost (euros):", rw.total_cost_eur.sum())
print("paid to invitees with 0 splits (euros):", (rw.invitee_activated == 0).sum() * 10,
      "=", round((rw.invitee_activated == 0).mean() * 100, 1), "% of the budget")

The bonus cost EUR 165,330 and 37% (EUR 61,340) went to invitees who never made a split, against the rule. The leak was 53% in Dec 2025 and fell to about 21% in May 2026, so something changed in the process. We should ask engineering.

## 4. Is the viral coefficient really close to 1?

K = invites sent per user x conversion of an invite into a new user. Equivalent and easier to audit: K = new users brought by a cohort / size of that cohort.

We do not know the exact formula the team used, so first we reproduce the most common shortcut and then correct it step by step.

In [ ]:
# step 1: the shortcut, invites per INVITER x acceptance, on the raw file
inviters_raw = invites_raw.inviter_user_id.nunique()
invites_per_inviter = len(invites_raw) / inviters_raw
accept_raw = invites_raw.accepted.mean()
K_headline = invites_per_inviter * accept_raw
print("invites per inviter:", round(invites_per_inviter, 2), "| acceptance:", round(accept_raw, 3))
print("K headline:", round(K_headline, 2))

In [ ]:
# only some users invite anybody
senders = users[users.user_id.isin(invites.inviter_user_id)]
print("share of users who ever sent an invite:", round(len(senders) / len(users), 3))
print("share of inviters that are activated:", round(senders.activated.mean(), 3))

# step 2: divide by all users, clean data
K_per_user = len(invites) / len(users) * invites.accepted.mean()
print("K per user, clean data:", round(K_per_user, 3))

The shortcut gives K = 5.6 x 20.4% = 1.14. It is wrong for three reasons: it divides by inviters only (just 37.6% of users invite), it counts duplicated and QA invites, and it counts signups that never use the app. Dividing by all users with clean data gives 0.43.

### 4.1 K by inviter cohort

In [ ]:
inviter_info = users[["user_id", "signup_month", "signup_date"]].rename(columns={
    "user_id": "inviter_user_id", "signup_month": "inviter_cohort", "signup_date": "inviter_signup"})
invited = users[(users.source == "invite") & (users.signup_date <= LAST_FULL)].merge(
    inviter_info, on="inviter_user_id", how="inner")
invited["delay_days"] = (invited.signup_date - invited.inviter_signup).dt.days
display(invited.delay_days.describe().to_frame().T)

Invitees sign up 14 days on average after the inviter (max 29), so cohorts up to April 2026 are complete.

In [ ]:
LAST_COHORT = pd.Period("2026-04")
cohort_size = users[users.signup_date <= LAST_FULL].groupby("signup_month").size().rename("cohort_size")

k = invited.groupby("inviter_cohort").agg(invitees=("user_id", "count"), activated=("activated", "sum"))
k["good"] = invited[invited.good_user == 1].groupby("inviter_cohort").size()
k = k.join(cohort_size)
k = k[k.index <= LAST_COHORT]
k["K_all"] = k.invitees / k.cohort_size
k["K_activated"] = k.activated / k.cohort_size
k["K_good"] = k.good / k.cohort_size
display(k)

In [ ]:
k[["K_all", "K_activated", "K_good"]].plot(figsize=(10, 4), title="Viral coefficient by inviter cohort")
plt.axhline(1, color="red", ls="--")
plt.axvline(pd.Period("2025-11"), color="gray", ls="--")
plt.ylim(0, 1.1)
plt.show()

In [ ]:
k["period"] = get_period(k.index.to_timestamp())
k_period = k.groupby("period")[["invitees", "activated", "good", "cohort_size"]].sum()
for c in ["invitees", "activated", "good"]:
    k_period["K_" + c] = k_period[c] / k_period.cohort_size
display(k_period)

In [ ]:
ladder = pd.DataFrame({
    "step": ["1 headline (per inviter, raw)", "2 per user, clean", "3 cohort K, all invitees",
             "4 cohort K, activated invitees", "5 cohort K, good invitees"],
    "K": [K_headline, K_per_user, k_period.loc["3_referral_plus_sync", "K_invitees"],
          k_period.loc["3_referral_plus_sync", "K_activated"], k_period.loc["3_referral_plus_sync", "K_good"]]})
display(ladder)

ladder.set_index("step").K.plot(kind="barh", figsize=(9, 3.5), title="K ladder: every correction lowers K")
plt.axvline(1, color="red", ls="--")
plt.show()

| Measure | Before bonus | Bonus | Bonus + sync |
|---|---|---|---|
| K, all invitees | 0.21 | 0.61 | 0.64 |
| K, activated invitees | 0.12 | 0.18 | 0.20 |
| K, good invitees | 0.059 | 0.044 | 0.048 |

* The bonus tripled the raw K but K of good users did not improve (0.059 to 0.048). It bought signups, not real users.
* Even the most generous measure (0.64) is below 1. K is not close to 1.
* With K = 0.2 the loop multiplies acquisition by 1/(1-0.2) = 1.25x. With K = 0.05 it is 1.05x. The loop is a booster, not an engine.

### 4.2 K by channel

In [ ]:
acc_channel = invites[invites.accepted == 1][["invitee_user_id", "channel"]]
invited_ch = invited.merge(acc_channel, left_on="user_id", right_on="invitee_user_id", how="left")
invited_ch = invited_ch[(invited_ch.inviter_cohort >= pd.Period("2025-11")) & (invited_ch.inviter_cohort <= LAST_COHORT)]
size_ref = cohort_size[(cohort_size.index >= pd.Period("2025-11")) & (cohort_size.index <= LAST_COHORT)].sum()

k_ch = invited_ch.groupby("channel").agg(invitees=("user_id", "count"), activated=("activated", "sum"), good=("good_user", "sum"))
for c in ["invitees", "activated", "good"]:
    k_ch["K_" + c] = k_ch[c] / size_ref
k_ch["good_share_of_signups"] = k_ch.good / k_ch.invitees
display(k_ch)

WhatsApp produces 63% of the good user K, link 17%, SMS 13% and auto SMS only 7%. Link brings many invitees but only 3.6% are good (9.5% for WhatsApp). Auto SMS only affects the Mar and Apr 2026 cohorts so far, so its weight will grow, but its quality is already visible.

## 5. Channels, segments, countries and periods: does it survive like with like?

In [ ]:
def seg_table(df, by):
    out = df.groupby(by).agg(users=("user_id", "count"), activation=("activated", "mean"))
    d30 = df[df.obs_30]
    out["retained_30"] = d30.groupby(by).retained_30.mean()
    out["good_rate"] = d30.groupby(by).good_user.mean()
    out["splits_if_activated"] = df[df.activated == 1].groupby(by).splits_first_30d.mean()
    return out

display(seg_table(full, ["period", "source"]))

In [ ]:
# does the country explain it? activation by source and country inside each period
display(full.groupby(["period", "source", "country"]).activated.mean().unstack("country"))
display(full[full.source == "invite"].groupby("period").country.value_counts(normalize=True).unstack())

* Paid and organic are stable and almost identical (activation 54 to 56%, retained at day 30 about 23 to 24%).
* Invitees before the bonus were the best users (activation 62%, 3.0 splits). After the bonus they are the worst (29 to 32%, good rate 7 to 7.7%).
* Countries explain nothing: inside each source and period, FR, ES, IT and DE differ by at most 3 points, even if the country mix of invitees changed a lot.

In [ ]:
# like with like: same invite channel before and after the bonus
like = acc.merge(inv_full[["invite_id", "period"]], on="invite_id").groupby(["channel", "period"]).agg(
    signups=("user_id", "count"), activation=("activated", "mean"))
display(like.unstack("period"))

In [ ]:
# reward audit: reward and first split should go together
post = full[(full.source == "invite") & (full.signup_date >= REFERRAL_START)].copy()
post["rewarded"] = post.user_id.isin(rewards.invitee_user_id)
audit = pd.crosstab(post.activated.map({1: "made a split", 0: "no split"}),
                    post.rewarded.map({True: "reward paid", False: "no reward"}))
display(audit)

It is not a channel mix effect. For the same channel, activation falls after the bonus (WhatsApp 62% to 38%, SMS 61% to 39%, link 63% to 16%). The incentive changed who signs up. The audit shows 5,938 rewards without a split and 137 splits without reward. Hypothesis for the A/B test and the fraud team: some signups come only for the EUR 5.

## 6. Unit economics: CAC, retention, LTV, payback

### 6.1 Monthly table (also used by the growth model)

In [ ]:
months = pd.period_range("2025-01", "2026-05", freq="M")
monthly = pd.DataFrame(index=months)
for s in ["paid", "organic", "invite"]:
    monthly[s + "_new"] = full[full.source == s].groupby("signup_month").size()
    monthly[s + "_act"] = full[full.source == s].groupby("signup_month").activated.sum()
monthly["new_total"] = monthly[["paid_new", "organic_new", "invite_new"]].sum(axis=1)

spend["pm"] = spend.month.dt.to_period("M")
rewards["pm"] = rewards.paid_date.dt.to_period("M")
monthly["spend"] = spend.set_index("pm").spend_eur
monthly["reward_eur"] = rewards.groupby("pm").total_cost_eur.sum()
monthly["reward_eur"] = monthly["reward_eur"].fillna(0)

In [ ]:
# active base: activated users already signed up and with activity on or after the 1st of the month
active = []
for m in months:
    first_day, last_day = m.start_time, m.end_time.normalize()
    active.append(((users.activated == 1) & (users.signup_date <= last_day) & (users.last_active_date >= first_day)).sum())
monthly["active"] = active

In [ ]:
monthly["cac_paid"] = monthly.spend / monthly.paid_new
monthly["cac_paid_per_activated"] = monthly.spend / monthly.paid_act
monthly["referral_cost_per_invitee"] = monthly.reward_eur / monthly.invite_new
monthly["blended_cac"] = (monthly.spend + monthly.reward_eur) / monthly.new_total
display(monthly)

monthly[["cac_paid", "blended_cac"]].plot(figsize=(10, 3.5), title="CAC per signup (euros)")
plt.axvline(pd.Period("2025-11"), color="gray", ls="--")
plt.show()

* Paid CAC is flat: EUR 3.20 per signup (EUR 5.9 per activated user) in all 18 months, no sign of saturation.
* Blended CAC jumped from EUR 1.35 to EUR 3.3 to 3.9 after the bonus. Each invitee costs about EUR 4.5 to 5.4 in rewards, more than a paid signup.
* Per good user: paid about EUR 14, referral about EUR 60 to 68. The free viral growth is 4 to 5 times more expensive than ads per retained user.

### 6.2 Retention curves

S(k) is the share of activated users still active k x 30 days after signup. To be fair we only use users who signed up by 2025-12-31 (all have 180 days of history).

In [ ]:
SEG_END = pd.Timestamp("2025-12-31")

def make_segment(df):
    return np.select([df.source == "paid", df.source == "organic",
                      (df.source == "invite") & (df.signup_date < REFERRAL_START)],
                     ["paid", "organic", "invite_before_bonus"], default="invite_after_bonus")

all_seg_users = users[users.signup_date <= SEG_END].copy()
all_seg_users["segment"] = make_segment(all_seg_users)
act_users = all_seg_users[all_seg_users.activated == 1]

In [ ]:
curve = pd.DataFrame({"S(0)": 1.0}, index=act_users.segment.unique())
for kk in range(1, 7):
    curve[f"S({kk})"] = act_users.groupby("segment").life_days.apply(lambda s: (s >= 30 * kk).mean())
display(curve)

curve.T.plot(figsize=(9, 4), marker="o", title="Retention of activated users")
plt.show()

In [ ]:
# cohort heatmap for 2025 cohorts
cohort = pd.DataFrame(index=pd.period_range("2025-01", "2025-12", freq="M"), columns=range(1, 7), dtype=float)
a_users = users[users.activated == 1]
for m in cohort.index:
    d = a_users[a_users.signup_month == m]
    for kk in cohort.columns:
        cohort.loc[m, kk] = (d.life_days >= 30 * kk).mean()

plt.figure(figsize=(7, 4))
plt.imshow(cohort.values.astype(float), aspect="auto", cmap="Blues")
plt.yticks(range(len(cohort)), cohort.index.astype(str))
plt.xticks(range(6), [f"M{c}" for c in cohort.columns])
plt.colorbar(label="share still active")
plt.title("Cohort retention (activated users, 2025)")
plt.show()

* Still active after 1 and 6 months: paid 42% and 25%, organic 43% and 26%, invite before bonus 49% and 29%, invite after bonus 25% and 15%.
* Bonus invitees lose three quarters of the users in the first 30 days. After that the decay is similar for everybody (8 to 10% per month), so the damage is at the start: onboarding and incentive quality.
* Jan to Oct 2025 cohorts are very stable, so no strong seasonality. The drop starts with the Nov to Dec 2025 cohorts.

### 6.3 LTV, CAC and payback

Splitly does not charge anything yet, so LTV needs an assumption: revenue per split. We also compute the break-even revenue per split, which does not depend on it.

* ARPU per active month = revenue per split x average splits of an activated user in the first 30 days.
* Expected active months = sum of the retention curve. Observed for 6 months, extended to 24 keeping the monthly decay seen between month 3 and 6 (assumption).
* LTV per signup = activation x ARPU x expected active months.
* CAC per signup: paid = spend / paid signups (2025), invite after bonus = reward euros / invitees, organic and invite before bonus = 0.

In [ ]:
REV_PER_SPLIT = 0.50    # assumption in euros per split, it is NOT in the data

seg = act_users.groupby("segment").agg(avg_splits_active=("splits_first_30d", "mean"))
seg["signups"] = all_seg_users.groupby("segment").size()
seg["activation"] = all_seg_users.groupby("segment").activated.mean()

In [ ]:
# retention curve extended to 24 months
surv = {}
for name in curve.index:
    s_list = [1.0] + [curve.loc[name, f"S({kk})"] for kk in range(1, 7)]
    decay = (s_list[6] / s_list[3]) ** (1 / 3)
    for kk in range(7, 24):
        s_list.append(s_list[-1] * decay)
    surv[name] = np.array(s_list)

seg["active_months_6m"] = [surv[n][:6].sum() for n in seg.index]
seg["active_months_24m"] = [surv[n][:24].sum() for n in seg.index]

In [ ]:
paid_spend_2025 = spend[spend.month < "2026-01-01"].spend_eur.sum()
paid_signups_2025 = (all_seg_users.segment == "paid").sum()
inv_after = all_seg_users[all_seg_users.segment == "invite_after_bonus"]
reward_for_them = rewards[rewards.invitee_user_id.isin(inv_after.user_id)].total_cost_eur.sum()

seg["cac_per_signup"] = 0.0
seg.loc["paid", "cac_per_signup"] = paid_spend_2025 / paid_signups_2025
seg.loc["invite_after_bonus", "cac_per_signup"] = reward_for_them / len(inv_after)
seg["cac_per_activated"] = seg.cac_per_signup / seg.activation

In [ ]:
seg["arpu_month"] = REV_PER_SPLIT * seg.avg_splits_active
seg["ltv_6m_per_signup"] = seg.activation * seg.arpu_month * seg.active_months_6m
seg["ltv_24m_per_signup"] = seg.activation * seg.arpu_month * seg.active_months_24m
has_cac = seg.cac_per_signup > 0
seg["ltv_to_cac_24m"] = np.where(has_cac, seg.ltv_24m_per_signup / seg.cac_per_signup, np.nan)

def payback(name):
    cac = seg.loc[name, "cac_per_activated"]
    if cac == 0:
        return "no direct cost"
    cum = np.cumsum(seg.loc[name, "arpu_month"] * surv[name])   # cumulative revenue of one activated user
    reached = np.where(cum >= cac)[0]
    return str(reached[0] + 1) if len(reached) > 0 else ">24"

seg["payback_months"] = [payback(n) for n in seg.index]

# revenue per split needed for LTV(24m) = 3 x CAC, independent of our assumption
seg["rev_per_split_for_3x"] = np.where(has_cac,
    3 * seg.cac_per_signup / (seg.activation * seg.avg_splits_active * seg.active_months_24m), np.nan)
display(seg.T)

In [ ]:
# sensitivity of LTV:CAC (24m) to the revenue assumption
rows = []
for rps in [0.15, 0.30, 0.50, 1.00, 2.00]:
    ltv = seg.activation * rps * seg.avg_splits_active * seg.active_months_24m
    rows.append(pd.Series(np.where(has_cac, ltv / seg.cac_per_signup, np.nan), index=seg.index, name=f"{rps} eur/split"))
display(pd.DataFrame(rows))

With EUR 0.50 per split (about 1.7% of a EUR 30 split):

| Segment | CAC per signup | LTV 24m per signup | LTV:CAC | Payback |
|---|---|---|---|---|
| Paid | 3.20 | 3.62 | 1.1 | 15 months |
| Invite after bonus | 5.24 | 1.00 | 0.19 | more than 24 months |
| Invite before bonus | 0 | 4.99 | n/a | n/a |
| Organic | 0 | 3.86 | n/a | n/a |

* Paid is barely above break-even and referral after the bonus destroys value.
* For LTV:CAC = 3 we need about EUR 1.33 per split for paid and EUR 7.84 for bonus invitees. The second is unrealistic, so the bonus design must change, not only the monetisation.
* Caveats: revenue per split is an assumption, splits are those of the first 30 days, retention after month 6 is extrapolated.

## 7. What changed and what did it cause?

Two changes: the Nov 2025 referral bonus and the Mar 2026 contact sync. A before and after comparison does not prove causality, but paid and organic did not change, so they act as a control.

In [ ]:
ev = pd.DataFrame(index=monthly.index)
ev["invites_sent"] = inv_full.groupby("sent_month").size()
ev["auto_sms_invites"] = inv_full[inv_full.channel == "auto_sms_contact_sync"].groupby("sent_month").size()
ev["auto_sms_invites"] = ev.auto_sms_invites.fillna(0)
ev["invited_signups"] = monthly.invite_new
ev["accept_rate"] = ev.invited_signups / ev.invites_sent
ev["invitee_activation"] = monthly.invite_act / monthly.invite_new
ev["paid_signups"] = monthly.paid_new
ev["organic_signups"] = monthly.organic_new
ev["reward_eur"] = monthly.reward_eur

In [ ]:
r = ratings.copy()
r["m"] = r.month.dt.to_period("M")
r = r.set_index("m")
ev["store_rating"] = r.avg_store_rating
ev["n_reviews"] = r.n_reviews
ev["share_1_star"] = r.share_1_star
ev["period"] = get_period(ev.index.to_timestamp())
display(ev)
display(ev.groupby("period").mean(numeric_only=True).T)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ev[["invites_sent", "auto_sms_invites"]].plot(ax=ax[0], title="Invites sent per month")
ev[["store_rating"]].plot(ax=ax[1], color="tab:red", title="Store rating (1 star share on the right axis)")
ev.share_1_star.plot(ax=ax[1], secondary_y=True, color="gray")
for a in ax:
    a.axvline(pd.Period("2025-11"), color="gray", ls="--")
    a.axvline(pd.Period("2026-03"), color="red", ls="--")
plt.show()

In [ ]:
# good users from invites and the cost of the bonus
obs = full[full.obs_30]
imp = pd.DataFrame({
    "good_users_invite": obs[obs.source == "invite"].groupby("signup_month").good_user.sum(),
    "good_users_paid": obs[obs.source == "paid"].groupby("signup_month").good_user.sum()})
imp["reward_eur"] = monthly.reward_eur
imp["reward_eur_per_good_invitee"] = imp.reward_eur / imp.good_users_invite
imp["paid_eur_per_good_paid_user"] = monthly.spend / imp.good_users_paid
imp["period"] = ev.period
display(imp.dropna(subset=["good_users_invite"]))
display(imp.groupby("period").mean(numeric_only=True))

| | Nov 2025 bonus | Mar 2026 contact sync |
|---|---|---|
| Invites per month | 2.6k to 10.2k | 10.2k to 39.0k (67% auto SMS) |
| Acceptance | 25% to 39% | 39% to 14.5% |
| Invited signups per month | 650 to 4,000 | 4,000 to 5,500 |
| Invitee activation | 62% to 29% | 29% to 32% |
| Good invitees per month | 195 to 282 | 282 to 425 |
| Reward cost per month | 0 to EUR 19.7k | EUR 25.7k |
| Store rating | 4.59 to 4.43 | 4.43 to 3.54 |
| 1 star share | 7% to 12% | 12% to 42% |

* Bonus: x6 invited signups, but activation halved and about EUR 68 per good invitee.
* Contact sync: about +1,500 invited signups and +140 good users per month, but 26k automatic SMS per month at 2.5% acceptance, and the rating fell 0.9 stars in one month (Feb 4.52 to Mar 3.54).
* Paid and organic signups have not suffered yet, but a 3.5 star rating is a risk for store conversion.

## 8. North Star Metric

Candidates: signups (vanity), invites sent (inflated by auto SMS), K, active users, or a quality measure. Our proposal: good users per month, meaning new users who made a first split and are still active 30 days later. If the metric is good it should not double just because we send more SMS.

In [ ]:
nsm = pd.DataFrame(index=months)
nsm["signups"] = monthly.new_total
nsm["invites_sent"] = ev.invites_sent
nsm["good_users"] = full[full.obs_30].groupby("signup_month").good_user.sum()
nsm["good_per_100_signups"] = 100 * nsm.good_users / nsm.signups
display(nsm)

In [ ]:
idx = nsm[["signups", "invites_sent", "good_users"]] / nsm.loc[pd.Period("2025-06"), ["signups", "invites_sent", "good_users"]] * 100
idx.plot(figsize=(10, 4), title="Vanity metrics vs NSM (June 2025 = 100)")
plt.axvline(pd.Period("2025-11"), color="gray", ls="--")
plt.axvline(pd.Period("2026-03"), color="red", ls="--")
plt.show()

From June 2025 to May 2026 signups grew 154% and invites sent x14.5, but good users only 49% (868 to 1,291). Good users per 100 signups fell from 25 to about 14.

Why it is the right metric: it measures value delivered, combines activation and retention (where the funnel leaks), is not inflated by the bonus or auto SMS, feeds the loop (only activated users invite), and the whole team can move it. Companion metrics: K of good users, reward euros per good invitee, and store rating as a guardrail. Counter metric against tunnel vision: paid CAC per good user.

## 9. Growth model, backtest and 3 scenarios

### 9.1 Model design

Each month: paid users = spend / CAC, organic users grow at a fixed rate, invited users = v x active base (v is invited signups per active user), new activated users = each source x its activation rate, and active base = last active x (1 - churn) + new activated. Parameters are averages of the last 3 training months.

K of the model = v x act_inv / churn: activated users that one active user brings during its life. If K is 1 or more the loop sustains itself without ads.

In [ ]:
monthly["new_activated"] = monthly.paid_act + monthly.organic_act + monthly.invite_act
monthly["churn"] = (monthly.active.shift(1) + monthly.new_activated - monthly.active) / monthly.active.shift(1)
monthly["v"] = monthly.invite_new / monthly.active.shift(1)
monthly["org_growth"] = monthly.organic_new.pct_change()
display(monthly[["active", "new_activated", "churn", "v", "org_growth"]])

In [ ]:
def get_params(train_months):
    t = monthly.loc[train_months]
    return {
        "cac_paid": t.spend.sum() / t.paid_new.sum(),
        "act_paid": t.paid_act.sum() / t.paid_new.sum(),
        "act_org": t.organic_act.sum() / t.organic_new.sum(),
        "act_inv": t.invite_act.sum() / t.invite_new.sum(),
        "v": t.v.mean(),
        "churn": t.churn.mean(),
        "org_growth": t.org_growth.mean(),
        "reward_per_invitee": t.reward_eur.sum() / t.invite_new.sum()}

In [ ]:
def simulate(p, start_active, start_organic, spend_list):
    rows = []
    A, org = start_active, start_organic
    for s in spend_list:
        paid = s / p["cac_paid"]
        org = org * (1 + p["org_growth"])
        inv = p["v"] * A
        new_act = paid * p["act_paid"] + org * p["act_org"] + inv * p["act_inv"]
        A = A * (1 - p["churn"]) + new_act
        rows.append({"spend": s, "paid_new": paid, "organic_new": org, "invite_new": inv,
                     "new_total": paid + org + inv, "new_activated": new_act, "active": A,
                     "reward_eur": inv * p["reward_per_invitee"]})
    return pd.DataFrame(rows)

In [ ]:
p_now = get_params(list(pd.period_range("2026-03", "2026-05", freq="M")))
display(pd.Series(p_now).to_frame("value"))
print("K of the model:", round(p_now["v"] * p_now["act_inv"] / p_now["churn"], 3))
print("loop needs v x act_inv >= churn. Today:", round(p_now["v"] * p_now["act_inv"], 3), "vs churn", round(p_now["churn"], 3))

Parameters (Mar to May 2026): paid CAC EUR 3.20, activation 56% (paid and organic) and 32% (invited), v = 0.48, monthly churn 27%, organic growth 3%. K of the model is 0.56 per active user. The loop is 1.8 times too weak (0.153 vs 0.273).

Warning: v was 0.52 in March, 0.51 in April and 0.40 in May, so invitation intensity is already fading and the base case may be a bit optimistic.

### 9.2 Backtest

We pretend to be at an earlier date, estimate parameters with the 3 previous months, and forecast the next months using the real marketing spend. We compare with a naive forecast that repeats the last training month.

* A: train Dec 2025 to Feb 2026, test Mar to May 2026 (includes the contact sync shock).
* B: train Jan to Mar 2026, test Apr to May 2026.

In [ ]:
def backtest(train_last, test_months):
    p = get_params([train_last - 2, train_last - 1, train_last])
    sim = simulate(p, monthly.active[train_last], monthly.organic_new[train_last], monthly.spend[test_months].tolist())
    sim.index = test_months
    real = monthly.loc[test_months]
    return pd.DataFrame({
        "new_total_real": real.new_total, "new_total_model": sim.new_total,
        "invite_new_real": real.invite_new, "invite_new_model": sim.invite_new,
        "active_real": real.active, "active_model": sim.active,
        "active_naive": monthly.active[train_last]})

def mape(real, pred):
    return float(np.mean(np.abs(real - pred) / real) * 100)

In [ ]:
tests = {"A": (pd.Period("2026-02"), list(pd.period_range("2026-03", "2026-05", freq="M"))),
         "B": (pd.Period("2026-03"), list(pd.period_range("2026-04", "2026-05", freq="M")))}
bt_rows = []
for name, (train_last, test_m) in tests.items():
    res = backtest(train_last, test_m)
    display(res.round(0))
    bt_rows.append({"backtest": name,
                    "MAPE new users": mape(res.new_total_real, res.new_total_model),
                    "MAPE invited users": mape(res.invite_new_real, res.invite_new_model),
                    "MAPE active (model)": mape(res.active_real, res.active_model),
                    "MAPE active (naive)": mape(res.active_real, res.active_naive)})
    res_B = res
display(pd.DataFrame(bt_rows).set_index("backtest"))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
monthly.active.plot(ax=ax[0], label="real", marker="o")
res_B.active_model.plot(ax=ax[0], label="model (backtest B)", marker="x", color="red")
ax[0].set_title("Active base: real vs backtest")
ax[0].legend()
monthly.invite_new.plot(ax=ax[1], label="real", marker="o")
res_B.invite_new_model.plot(ax=ax[1], label="model (backtest B)", marker="x", color="red")
ax[1].set_title("Invited signups: real vs backtest")
ax[1].legend()
plt.show()

The model predicts the active base very well (error 1.4% and 0.7%, vs 11.4% and 6.2% for the naive forecast). New users have 7% error. Invited signups have 12% error because the model overshoots May (5,713 vs 4,928): invitation intensity started to decline and constant parameters cannot see that. So the model is reliable for the active base and less for the invite flow, which is why the pessimistic scenario matters.

### 9.3 Three scenarios (Jun 2026 to May 2027)

All start from the real state of May 2026. Spend: real June 2026 value, then the historical monthly growth.

| Scenario | What changes versus base (Mar to May 2026 parameters) |
|---|---|
| Base | Nothing changes |
| Optimistic | Opt-in sync and reward after real use: v +20%, invitee activation x1.35, churn -10%, CAC -5% |
| Pessimistic | Rating damage and ad fatigue: CAC +25%, v -30%, invitee activation -10%, churn +15% |

In [ ]:
spend_growth = spend.spend_eur.pct_change().tail(6).mean()
print("monthly spend growth used:", round(spend_growth, 4))

fut_months = pd.period_range("2026-06", "2027-05", freq="M")
june_spend = spend[spend.pm == pd.Period("2026-06")].spend_eur.iloc[0]
spend_future = [june_spend * (1 + spend_growth) ** i for i in range(12)]

scenarios = {
    "Base": {},
    "Optimistic": {"v": 1.20, "act_inv": 1.35, "churn": 0.90, "cac_paid": 0.95},
    "Pessimistic": {"cac_paid": 1.25, "v": 0.70, "act_inv": 0.90, "churn": 1.15}}

In [ ]:
results, sc_rows = {}, []
for name, mult in scenarios.items():
    p = dict(p_now)
    for key, factor in mult.items():
        p[key] = p[key] * factor
    sim = simulate(p, monthly.active[pd.Period("2026-05")], monthly.organic_new[pd.Period("2026-05")], spend_future)
    sim.index = fut_months
    results[name] = sim
    total_cost = sim.spend.sum() + sim.reward_eur.sum()
    sc_rows.append({"scenario": name,
                    "K per active user": p["v"] * p["act_inv"] / p["churn"],
                    "v x act_inv": p["v"] * p["act_inv"],
                    "churn": p["churn"],
                    "active base May-27": sim.active.iloc[-1],
                    "new users per month May-27": sim.new_total.iloc[-1],
                    "share of new users from invites": sim.invite_new.iloc[-1] / sim.new_total.iloc[-1],
                    "12m cost EUR": total_cost,
                    "cost per new activated EUR": total_cost / sim.new_activated.sum()})
sc = pd.DataFrame(sc_rows).set_index("scenario")
display(sc.T.round(2))

In [ ]:
colors = {"Base": "tab:blue", "Optimistic": "tab:green", "Pessimistic": "tab:red"}
plt.figure(figsize=(10, 4.5))
monthly.active.plot(label="real", color="black", marker="o")
for name, sim in results.items():
    sim.active.plot(label=name, color=colors[name], marker=".")
plt.title("Active users: history and 3 scenarios")
plt.legend()
plt.show()

In [ ]:
# what would it take for K = 1?
need_v = p_now["churn"] / p_now["act_inv"]
print(f"For K = 1 we need v of {need_v:.2f} invited signups per active user per month, today v = {p_now['v']:.2f}")
print(f"The loop must be {need_v / p_now['v']:.1f}x stronger than today")

| | Base | Optimistic | Pessimistic |
|---|---|---|---|
| K per active user | 0.56 | 1.01 | 0.31 |
| Active base May 2027 | 20.1k (+58%) | 44.8k (x3.5) | 11.4k (-11%) |
| New users per month May 2027 | 14.7k | 29.3k | 8.5k |
| Share of new users from invites | 63% | 81% | 44% |
| 12 month cost | EUR 523k | EUR 940k | EUR 299k |
| Cost per new activated user | EUR 8.7 | EUR 8.5 | EUR 7.5 |

* Base: growth continues but it is driven by ads and organic. The loop adds, it does not sustain.
* Optimistic: K reaches about 1, and even then it is borderline. It needs invitation intensity about 1.8x today.
* Pessimistic: the active base shrinks.
* Levers: channel mix (shift reward budget to WhatsApp), onboarding (biggest leak), reward design (pay after real use), monetisation (to reach LTV:CAC of 3).

## 10. A/B test plan

Hypothesis: the opt-out auto SMS contact sync creates volume but low quality invites and hurts the rating. Making it opt-in (and paying the reward after the invitee's 2nd split) will reduce invites but keep or raise good invitees, and protect the rating.

| Item | Proposal |
|---|---|
| Randomisation | New user at onboarding, 50/50, stratified by country |
| Control | Contact sync ON by default (today) |
| Treatment | Contact sync OFF by default (explicit opt-in screen) |
| Primary metric | Share of new users who bring at least 1 activated invitee within 30 days |
| Secondary | Invites per user, accept rate, good invitees per 100 new users, reward euros per good invitee |
| Guardrails | 1 star share, uninstall rate in first 7 days, SMS complaints and opt-outs |
| Decision rule | Ship if the primary metric is not worse (margin 1 point) and guardrails improve |

In [ ]:
from statistics import NormalDist

# baseline: share of users (cohorts Mar and Apr 2026) with at least one activated invitee
cohort_users = users[(users.signup_month >= pd.Period("2026-03")) & (users.signup_month <= LAST_COHORT)]
act_invitees = invited[invited.activated == 1]
baseline = cohort_users.user_id.isin(act_invitees.inviter_user_id).mean()
print("baseline share:", round(baseline, 4))

In [ ]:
def n_per_arm(p1, p2, alpha=0.05, power=0.80):
    za = NormalDist().inv_cdf(1 - alpha / 2)
    zb = NormalDist().inv_cdf(power)
    pbar = (p1 + p2) / 2
    n = (za * (2 * pbar * (1 - pbar)) ** 0.5 + zb * (p1 * (1 - p1) + p2 * (1 - p2)) ** 0.5) ** 2 / (p1 - p2) ** 2
    return int(np.ceil(n))

days = (pd.Timestamp("2026-04-30") - pd.Timestamp("2026-03-01")).days + 1
new_users_per_day = len(cohort_users) / days

rows = []
for mde in [0.005, 0.01, 0.02]:
    n = n_per_arm(baseline, baseline + mde)
    rows.append({"detectable effect (points)": mde * 100, "users per arm": n,
                 "days needed": np.ceil(2 * n / new_users_per_day) + 30})   # +30 days to observe the outcome
display(pd.DataFrame(rows))
print("new users per day:", round(new_users_per_day))

## 11. Answer to the board

Is Splitly's viral coefficient really close to 1? No.

1. The "close to 1" comes from a shortcut (invites per inviter x acceptance = 1.14) that divides by the 38% of users who invite and counts duplicates, QA accounts and signups that never use the app.
2. Clean K per signup is 0.64 counting every invitee, 0.20 counting activated invitees and 0.05 counting good users. The bonus tripled the first number and did not improve the last one.
3. The loop is a booster (about 1.05 to 1.25x), not an engine. Growth is still paid by ads and organic (about 40% of new users).

What went wrong:
* Value leaks in three places: acceptance after auto SMS (2.5%), invitee activation (62% to 30%) and 30 day retention of invitees (49% to 24%).
* Referral is 4 to 5 times more expensive than paid per retained user (EUR 60 to 68 vs EUR 14), and 37% of the reward budget (EUR 61k) went to invitees who never made a split.
* The contact sync gave +140 good users per month but cost 0.9 stars of rating (4.43 to 3.54) and doubled the reviews.

Recommendations, in order:
1. Pay the reward only after real use (2nd split) and audit the 6k rewards without a split.
2. Make the contact sync opt-in (A/B test in section 10), cap auto SMS and push WhatsApp invites.
3. Fix onboarding for invitees and track good users per month as the North Star.
4. Define monetisation: paid LTV:CAC is only about 1.1 with EUR 0.50 per split.

Limits: no revenue data (LTV is an assumption), no city data, before and after comparisons are not proof of causality, June 2026 was excluded, and model parameters use only 3 months.

Likely cold questions: Why not simply stop the bonus? What is K if we only count WhatsApp? How sure are we that the rating drop is caused by the sync and not by the bonus? What happens to paid CAC if the rating stays at 3.5?